In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass
print('done')
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

done


In [3]:
# Mango Leaf Disease Image Generation: VAE vs GAN
# Import necessary libraries

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid, save_image
from PIL import Image
from tqdm.notebook import tqdm
import random
from sklearn.model_selection import train_test_split
import torchvision.transforms.functional as TF
from scipy.linalg import sqrtm
import cv2
from skimage.metrics import structural_similarity as ssim

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Configuration
IMG_SIZE = 128
BATCH_SIZE = 32
NUM_EPOCHS_VAE = 50
NUM_EPOCHS_GAN = 100
LATENT_DIM = 128
LEARNING_RATE = 0.0002
BETA1 = 0.5
NUM_CLASSES = None  # Will be set after loading dataset
NUM_GENERATED_IMAGES = 100

Using device: cuda


In [4]:

# Custom Dataset class for mango leaf images
class MangoLeafDataset(Dataset):
    def __init__(self, img_paths, class_labels, transform=None):
        self.img_paths = img_paths
        self.class_labels = class_labels
        self.transform = transform
        
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.class_labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Function to load and preprocess the dataset
def load_dataset(data_dir, img_size=128, batch_size=32):
    # Define transformations
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    # Collect image paths and class labels
    img_paths = []
    class_labels = []
    class_to_idx = {}
    idx = 0
    
    # Assuming data structure: data_dir/class_name/image.jpg
    for class_name in os.listdir(data_dir):
        class_dir = os.path.join(data_dir, class_name)
        if os.path.isdir(class_dir):
            class_to_idx[class_name] = idx
            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(class_dir, img_name)
                    img_paths.append(img_path)
                    class_labels.append(idx)
            idx += 1
    
    # Split data into train and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        img_paths, class_labels, test_size=0.2, stratify=class_labels, random_state=42
    )
    
    # Create datasets
    train_dataset = MangoLeafDataset(train_paths, train_labels, transform)
    val_dataset = MangoLeafDataset(val_paths, val_labels, transform)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return train_loader, val_loader, idx, class_to_idx

In [5]:

# Define VAE architecture
class VAE(nn.Module):
    def __init__(self, latent_dim, num_classes):
        super(VAE, self).__init__()
        
        # Encoder
        self.enc_conv1 = nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1)
        self.enc_conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.enc_conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.enc_conv4 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        
        # Size after encoder: 256 x 8 x 8
        self.fc_mu = nn.Linear(256 * 8 * 8, latent_dim)
        self.fc_var = nn.Linear(256 * 8 * 8, latent_dim)
        
        # Decoder
        self.dec_fc = nn.Linear(latent_dim + num_classes, 256 * 8 * 8)
        self.dec_conv1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec_conv2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec_conv3 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.dec_conv4 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
        
        self.num_classes = num_classes
        
    def encode(self, x):
        x = F.relu(self.enc_conv1(x))
        x = F.relu(self.enc_conv2(x))
        x = F.relu(self.enc_conv3(x))
        x = F.relu(self.enc_conv4(x))
        x = x.view(x.size(0), -1)
        
        mu = self.fc_mu(x)
        log_var = self.fc_var(x)
        
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def decode(self, z, c=None):
        if c is not None:
            # One-hot encode the class label
            c_one_hot = F.one_hot(c, self.num_classes).float()
            z = torch.cat([z, c_one_hot], dim=1)
        
        x = self.dec_fc(z)
        x = x.view(x.size(0), 256, 8, 8)
        
        x = F.relu(self.dec_conv1(x))
        x = F.relu(self.dec_conv2(x))
        x = F.relu(self.dec_conv3(x))
        x = torch.tanh(self.dec_conv4(x))
        
        return x
    
    def forward(self, x, c=None):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_reconstructed = self.decode(z, c)
        
        return x_reconstructed, mu, log_var

In [6]:
# Define GAN architecture (DCGAN-based)
class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes):
        super(Generator, self).__init__()
        
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        
        self.fc = nn.Linear(latent_dim + num_classes, 256 * 8 * 8)
        
        self.main = nn.Sequential(
            # Initial size: 256 x 8 x 8
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # Size: 128 x 16 x 16
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # Size: 64 x 32 x 32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            # Size: 32 x 64 x 64
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
            # Final size: 3 x 128 x 128
        )
    
    def forward(self, z, c=None):
        if c is not None:
            # One-hot encode the class label
            c_one_hot = F.one_hot(c, self.num_classes).float()
            z = torch.cat([z, c_one_hot], dim=1)
        
        x = self.fc(z)
        x = x.view(x.size(0), 256, 8, 8)
        x = self.main(x)
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        
        self.main = nn.Sequential(
            # Input: 3 x 128 x 128
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Size: 32 x 64 x 64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            # Size: 64 x 32 x 32
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # Size: 128 x 16 x 16
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # Size: 256 x 8 x 8
            nn.Conv2d(256, 1, kernel_size=8, stride=1, padding=0, bias=False),
            # Size: 1 x 1 x 1
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = self.main(x)
        return x.view(-1, 1).squeeze(1)


In [7]:

# Function to train VAE
def train_vae(vae, train_loader, val_loader, num_epochs, device):
    optimizer = optim.Adam(vae.parameters(), lr=LEARNING_RATE)
    
    # Lists to store metrics
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # Training
        vae.train()
        running_loss = 0.0
        
        for batch_idx, (imgs, labels) in enumerate(tqdm(train_loader, desc=f"VAE Epoch {epoch+1}/{num_epochs}")):
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            # Forward pass
            reconstructed_imgs, mu, log_var = vae(imgs, labels)
            
            # Reconstruction loss
            reconstruction_loss = F.mse_loss(reconstructed_imgs, imgs, reduction='sum')
            
            # KL divergence loss
            kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
            
            # Total loss
            loss = reconstruction_loss + kl_loss
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        
        # Validation
        vae.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_idx, (imgs, labels) in enumerate(val_loader):
                imgs = imgs.to(device)
                labels = labels.to(device)
                
                # Forward pass
                reconstructed_imgs, mu, log_var = vae(imgs, labels)
                
                # Reconstruction loss
                reconstruction_loss = F.mse_loss(reconstructed_imgs, imgs, reduction='sum')
                
                # KL divergence loss
                kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
                
                # Total loss
                loss = reconstruction_loss + kl_loss
                val_loss += loss.item()
            
            val_loss /= len(val_loader.dataset)
            val_losses.append(val_loss)
            
            # Print metrics
            print(f"VAE Epoch: {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.6f}, Val Loss: {val_loss:.6f}")
            
            # Visualize reconstructions after certain epochs
            if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
                visualize_reconstructions(vae, val_loader, device, epoch+1)
    
    # Plot training and validation loss
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE Training and Validation Loss')
    plt.legend()
    plt.savefig('vae_loss_plot.png')
    plt.show()
    
    return vae

# Function to visualize VAE reconstructions
def visualize_reconstructions(vae, data_loader, device, epoch):
    vae.eval()
    with torch.no_grad():
        # Get a batch of images
        imgs, labels = next(iter(data_loader))
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        # Reconstruct images
        reconstructed_imgs, _, _ = vae(imgs, labels)
        
        # Denormalize images
        imgs = (imgs + 1) / 2
        reconstructed_imgs = (reconstructed_imgs + 1) / 2
        
        # Display original and reconstructed images
        n = min(8, imgs.size(0))
        fig, axes = plt.subplots(2, n, figsize=(12, 4))
        
        for i in range(n):
            # Original images
            orig_img = imgs[i].cpu().permute(1, 2, 0).numpy()
            axes[0, i].imshow(orig_img)
            axes[0, i].axis('off')
            if i == 0:
                axes[0, i].set_title('Original')
            
            # Reconstructed images
            recon_img = reconstructed_imgs[i].cpu().permute(1, 2, 0).numpy()
            axes[1, i].imshow(recon_img)
            axes[1, i].axis('off')
            if i == 0:
                axes[1, i].set_title('Reconstructed')
        
        plt.tight_layout()
        plt.savefig(f'vae_reconstruction_epoch{epoch}.png')
        plt.show()

# Function to train GAN
def train_gan(generator, discriminator, train_loader, num_epochs, device, num_classes):
    # Optimizers
    optim_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
    optim_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
    
    # Loss function
    criterion = nn.BCELoss()
    
    # Lists to store metrics
    d_losses = []
    g_losses = []
    
    # Fixed noise for visualization
    fixed_noise = torch.randn(64, LATENT_DIM, device=device)
    fixed_labels = torch.randint(0, num_classes, (64,), device=device)
    
    for epoch in range(num_epochs):
        # Training
        running_d_loss = 0.0
        running_g_loss = 0.0
        
        for batch_idx, (real_imgs, labels) in enumerate(tqdm(train_loader, desc=f"GAN Epoch {epoch+1}/{num_epochs}")):
            batch_size = real_imgs.size(0)
            real_imgs = real_imgs.to(device)
            labels = labels.to(device)
            
            # Labels for real and fake images
            real_label = torch.ones(batch_size, device=device)
            fake_label = torch.zeros(batch_size, device=device)
            
            ### Train Discriminator ###
            optim_d.zero_grad()
            
            # Train with real images
            output_real = discriminator(real_imgs)
            d_loss_real = criterion(output_real, real_label)
            
            # Train with fake images
            noise = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_labels = torch.randint(0, num_classes, (batch_size,), device=device)
            fake_imgs = generator(noise, fake_labels)
            output_fake = discriminator(fake_imgs.detach())
            d_loss_fake = criterion(output_fake, fake_label)
            
            # Total discriminator loss
            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optim_d.step()
            
            ### Train Generator ###
            optim_g.zero_grad()
            
            # Generate new fake images
            noise = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_labels = torch.randint(0, num_classes, (batch_size,), device=device)
            fake_imgs = generator(noise, fake_labels)
            output_fake = discriminator(fake_imgs)
            
            # Generator tries to make discriminator think its images are real
            g_loss = criterion(output_fake, real_label)
            g_loss.backward()
            optim_g.step()
            
            # Update running losses
            running_d_loss += d_loss.item()
            running_g_loss += g_loss.item()
        
        # Calculate epoch losses
        epoch_d_loss = running_d_loss / len(train_loader)
        epoch_g_loss = running_g_loss / len(train_loader)
        
        # Store losses
        d_losses.append(epoch_d_loss)
        g_losses.append(epoch_g_loss)
        
        # Print metrics
        print(f"GAN Epoch: {epoch+1}/{num_epochs}, D Loss: {epoch_d_loss:.6f}, G Loss: {epoch_g_loss:.6f}")
        
        # Generate and save sample images
        if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            with torch.no_grad():
                fake_imgs = generator(fixed_noise, fixed_labels)
                fake_imgs = (fake_imgs + 1) / 2  # Denormalize
                
                # Create grid of images
                img_grid = make_grid(fake_imgs, nrow=8, normalize=False)
                save_image(img_grid, f"gan_samples_epoch{epoch+1}.png")
                
                # Display grid
                plt.figure(figsize=(10, 10))
                plt.imshow(img_grid.cpu().permute(1, 2, 0).numpy())
                plt.axis('off')
                plt.title(f"GAN Generated Images - Epoch {epoch+1}")
                plt.show()
    
    # Plot training losses
    plt.figure(figsize=(10, 5))
    plt.plot(d_losses, label='Discriminator Loss')
    plt.plot(g_losses, label='Generator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('GAN Training Losses')
    plt.legend()
    plt.savefig('gan_loss_plot.png')
    plt.show()
    
    return generator, discriminator

# Function to generate images from VAE
def generate_images_vae(vae, num_images, num_classes, device):
    vae.eval()
    generated_images = []
    generated_labels = []
    
    with torch.no_grad():
        for i in range(num_images):
            # Sample from latent space
            z = torch.randn(1, LATENT_DIM).to(device)
            
            # Randomly select a class
            label = torch.tensor([random.randint(0, num_classes-1)], device=device)
            
            # Generate image
            generated_img = vae.decode(z, label)
            
            # Denormalize
            generated_img = (generated_img + 1) / 2
            
            generated_images.append(generated_img.cpu())
            generated_labels.append(label.item())
    
    # Stack all images
    generated_images = torch.cat(generated_images, dim=0)
    
    return generated_images, generated_labels

# Function to generate images from GAN
def generate_images_gan(generator, num_images, num_classes, device):
    generator.eval()
    generated_images = []
    generated_labels = []
    
    with torch.no_grad():
        for i in range(num_images):
            # Sample from latent space
            z = torch.randn(1, LATENT_DIM).to(device)
            
            # Randomly select a class
            label = torch.tensor([random.randint(0, num_classes-1)], device=device)
            
            # Generate image
            generated_img = generator(z, label)
            
            # Denormalize
            generated_img = (generated_img + 1) / 2
            
            generated_images.append(generated_img.cpu())
            generated_labels.append(label.item())
    
    # Stack all images
    generated_images = torch.cat(generated_images, dim=0)
    
    return generated_images, generated_labels

# Function to calculate SSIM between two sets of images
# def calculate_ssim(real_images, generated_images):
#     ssim_scores = []
    
#     # Take a subset of real images if there are too many
#     real_subset = real_images[:len(generated_images)]
    
#     for i in range(len(generated_images)):
#         real_img = real_subset[i].permute(1, 2, 0).numpy()
#         gen_img = generated_images[i].permute(1, 2, 0).numpy()
        
#         # Calculate SSIM for multichannel images
#         score, _ = ssim(real_img, gen_img, full=True, multichannel=True)
#         ssim_scores.append(score)
    
#     return np.mean(ssim_scores)

# Function to calculate new code for SSIM between two sets of images
# Function to calculate SSIM between two sets of images
def calculate_ssim(real_images, generated_images):
    ssim_scores = []
    
    # Take a subset of real images if there are too many
    real_subset = real_images[:len(generated_images)]
    
    for i in range(len(generated_images)):
        real_img = real_subset[i].permute(1, 2, 0).numpy()
        gen_img = generated_images[i].permute(1, 2, 0).numpy()
        
        # Ensure window size is appropriate for the image dimensions
        min_dim = min(real_img.shape[0], real_img.shape[1])
        win_size = min(7, min_dim - 1)  # Make sure window size is odd and smaller than image
        if win_size % 2 == 0:
            win_size -= 1  # Ensure odd number
        
        # Calculate SSIM for multichannel images (explicitly set channel_axis)
        score = ssim(real_img, gen_img, win_size=win_size, channel_axis=2, data_range=1.0)
        ssim_scores.append(score)
    
    return np.mean(ssim_scores)  ##did not try

# Function to calculate Fréchet Inception Distance (FID)
def calculate_fid(real_features, generated_features):
    # Calculate mean and covariance for both feature sets
    mu1, sigma1 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = np.mean(generated_features, axis=0), np.cov(generated_features, rowvar=False)
    
    # Calculate sum of squared difference between means
    ssdiff = np.sum((mu1 - mu2) ** 2.0)
    
    # Calculate matrix sqrt
    covmean = sqrtm(sigma1.dot(sigma2))
    
    # Check and correct imaginary component if necessary
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    # Calculate FID
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    
    return fid

# Function to extract features using inception model
def extract_features(images, model):
    model.eval()
    features = []
    
    with torch.no_grad():
        for i in range(len(images)):
            # Add batch dimension
            img = images[i].unsqueeze(0)
            
            # Extract features (up to the last pooling layer)
            feature = model(img)
            features.append(feature.cpu().numpy().flatten())
    
    return np.array(features)

## ...main execution after this

In [8]:
##main execution
def main():
    print("Starting mango leaf disease image generation project...")
    
    # Path to the dataset (update this to your dataset path on Kaggle)
    data_dir = "/kaggle/input/mango-leaf-disease-dataset"  # Update with actual path
    
    # Load dataset
    print("Loading and preprocessing the dataset...")
    train_loader, val_loader, num_classes, class_to_idx = load_dataset(data_dir, IMG_SIZE, BATCH_SIZE)
    
    print(f"Dataset loaded. Number of classes: {num_classes}")
    print("Class mapping:", class_to_idx)
    
    # Initialize models
    print("Initializing VAE model...")
    vae = VAE(LATENT_DIM, num_classes).to(device)
    
    print("Initializing GAN model...")
    generator = Generator(LATENT_DIM, num_classes).to(device)
    discriminator = Discriminator().to(device)
    
    # Train VAE
    print("Training VAE...")
    vae = train_vae(vae, train_loader, val_loader, NUM_EPOCHS_VAE, device)
    
    # Save VAE model
    torch.save(vae.state_dict(), 'vae_model.pth')
    print("VAE model saved.")
    
    # Train GAN
    print("Training GAN...")
    generator, discriminator = train_gan(generator, discriminator, train_loader, NUM_EPOCHS_GAN, device, num_classes)
    
    # Save GAN models
    torch.save(generator.state_dict(), 'gan_generator.pth')
    torch.save(discriminator.state_dict(), 'gan_discriminator.pth')
    print("GAN models saved.")
    
    # Generate images from both models
    print(f"Generating {NUM_GENERATED_IMAGES} images from VAE...")
    vae_images, vae_labels = generate_images_vae(vae, NUM_GENERATED_IMAGES, num_classes, device)
    
    print(f"Generating {NUM_GENERATED_IMAGES} images from GAN...")
    gan_images, gan_labels = generate_images_gan(generator, NUM_GENERATED_IMAGES, num_classes, device)
    
    # Save generated images
    # VAE images
    os.makedirs('vae_generated', exist_ok=True)
    for i, (img, label) in enumerate(zip(vae_images, vae_labels)):
        save_image(img, f'vae_generated/vae_img_{i}_class_{label}.png')
    
    # GAN images
    os.makedirs('gan_generated', exist_ok=True)
    for i, (img, label) in enumerate(zip(gan_images, gan_labels)):
        save_image(img, f'gan_generated/gan_img_{i}_class_{label}.png')
    
    # Visualize generated images
    print("Visualizing generated images...")
    # VAE images grid
    vae_grid = make_grid(vae_images[:64], nrow=8, normalize=True)
    plt.figure(figsize=(15, 15))
    plt.imshow(vae_grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title('VAE Generated Images')
    plt.savefig('vae_generated_grid.png')
    plt.show()
    
    # GAN images grid
    gan_grid = make_grid(gan_images[:64], nrow=8, normalize=True)
    plt.figure(figsize=(15, 15))
    plt.imshow(gan_grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title('GAN Generated Images')
    plt.savefig('gan_generated_grid.png')
    plt.show()
    
    # Evaluation
    print("Evaluating models...")
    
    # Get real images for comparison
    real_images = []
    for imgs, _ in val_loader:
        real_images.extend([img for img in imgs])
        if len(real_images) >= NUM_GENERATED_IMAGES:
            break
    real_images = real_images[:NUM_GENERATED_IMAGES]
    real_images = [(img + 1) / 2 for img in real_images]  # Denormalize
    
    # Calculate SSIM
    print("Calculating SSIM...")
    vae_ssim = calculate_ssim(real_images, vae_images)
    gan_ssim = calculate_ssim(real_images, gan_images)
    
    print(f"VAE SSIM: {vae_ssim:.4f}")
    print(f"GAN SSIM: {gan_ssim:.4f}")
    
    # Calculate FID (need to extract features first)
    print("Calculating FID...")
    # Load pre-trained Inception model
    inception_model = models.inception_v3(pretrained=True, transform_input=True)
    # Remove last fully connected layer
    inception_model.fc = nn.Identity()
    inception_model = inception_model.to(device)
    
    # Extract features
    real_features = extract_features(torch.stack(real_images).to(device), inception_model)
    vae_features = extract_features(vae_images.to(device), inception_model)
    gan_features = extract_features(gan_images.to(device), inception_model)
    
    # Calculate FID
    vae_fid = calculate_fid(real_features, vae_features)
    gan_fid = calculate_fid(real_features, gan_features)
    
    print(f"VAE FID: {vae_fid:.4f}")
    print(f"GAN FID: {gan_fid:.4f}")
    
    # Comparison table
    metrics = {
        'Model': ['VAE', 'GAN'],
        'SSIM': [vae_ssim, gan_ssim],
        'FID': [vae_fid, gan_fid]
    }
    
    # Display comparison
    print("\nModel Comparison:")
    print(f"{'Model':<10}{'SSIM':<15}{'FID':<15}")
    print("-" * 40)
    print(f"{'VAE':<10}{vae_ssim:<15.4f}{vae_fid:<15.4f}")
    print(f"{'GAN':<10}{gan_ssim:<15.4f}{gan_fid:<15.4f}")
    
    # Plot comparative metrics
    plt.figure(figsize=(12, 5))
    
    # SSIM comparison (higher is better)
    plt.subplot(1, 2, 1)
    plt.bar(['VAE', 'GAN'], [vae_ssim, gan_ssim])
    plt.title('SSIM Comparison (Higher is Better)')
    plt.ylim(0, 1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # FID comparison (lower is better)
    plt.subplot(1, 2, 2)
    plt.bar(['VAE', 'GAN'], [vae_fid, gan_fid])
    plt.title('FID Comparison (Lower is Better)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig('model_comparison.png')
    plt.show()
    
    print("\nConclusion:")
    if vae_ssim > gan_ssim and vae_fid < gan_fid:
        print("VAE outperforms GAN in both metrics.")
    elif gan_ssim > vae_ssim and gan_fid < vae_fid:
        print("GAN outperforms VAE in both metrics.")
    else:
        if vae_ssim > gan_ssim:
            print("VAE produces images with higher structural similarity to real images.")
        else:
            print("GAN produces images with higher structural similarity to real images.")
        
        if vae_fid < gan_fid:
            print("VAE produces images with better feature distribution compared to real images.")
        else:
            print("GAN produces images with better feature distribution compared to real images.")
    
    print("\nMango Leaf Disease Image Generation project completed!")

if __name__ == "__main__":
    main()

Starting mango leaf disease image generation project...
Loading and preprocessing the dataset...
Dataset loaded. Number of classes: 8
Class mapping: {'Powdery Mildew': 0, 'Cutting Weevil': 1, 'Anthracnose': 2, 'Bacterial Canker': 3, 'Sooty Mould': 4, 'Gall Midge': 5, 'Healthy': 6, 'Die Back': 7}
Initializing VAE model...
Initializing GAN model...
Training VAE...


VAE Epoch 1/50:   0%|          | 0/100 [00:00<?, ?it/s]

VAE Epoch: 1/50, Train Loss: 8078.352476, Val Loss: 5043.930859


VAE Epoch 2/50:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 